In [2]:
from pathlib import Path

import pandas as pd


In [15]:
DATA = Path('./data/')

gse_me_hs = pd.read_csv(DATA / 'gse_me_hs.csv')
gse_me_mm = pd.read_csv(DATA / 'gse_me_mm.csv')
gse_rna_hs = pd.read_csv(DATA / 'gse_rna_hs.csv')
gse_rna_mm = pd.read_csv(DATA / 'gse_rna_mm.csv')

In [16]:
print(list(gse_me_hs.columns))
print(list(gse_me_mm.columns))
print(list(gse_rna_hs.columns))
print(list(gse_rna_mm.columns))

['title', 'description', 'organism', 'type', 'platform', 'accession', 'id', 'project', 'dataset', 'samples']
['title', 'description', 'organism', 'type', 'platform', 'accession', 'id', 'project', 'dataset', 'samples']
['title', 'description', 'organism', 'type', 'platform', 'accession', 'id', 'project', 'dataset', 'samples']
['title', 'description', 'organism', 'type', 'platform', 'accession', 'id', 'project', 'dataset', 'samples']


In [17]:
# Merge and save in one file in data directory
df = pd.concat([gse_me_hs, gse_me_mm, gse_rna_hs, gse_rna_mm], axis=0)

In [19]:
df['accession']

0        GSE228100
1        GSE243488
2        GSE243487
3        GSE226234
4        GSE226480
           ...    
51766        GSE85
51767        GSE36
51768        GSE30
51769        GSE11
51770         GSE2
Name: accession, Length: 116684, dtype: object

In [20]:
# Remove GSE which do not have any associated samples from data/GEO_** directories
"""
backend/data/
├── GEO_hs_dnam/
│   ├── clock_result/          ← много GSE*.csv (предсказания)
│   └── processed_metadata/    ← много GSE*.csv (метаданные)
├── GEO_hs_rna/
│   ├── clock_result/
│   └── processed_metadata/
├── GEO_mm_dnam/
│   ├── clock_result/
│   └── processed_metadata/
├── GEO_mm_rna/
│   ├── clock_result/
│   └── processed_metadata/

"""
valid_gse = set()
for subdir in ['GEO_hs_dnam', 'GEO_hs_rna', 'GEO_mm_dnam', 'GEO_mm_rna']:
    processed_metadata_dir = DATA / subdir / 'processed_metadata'
    for file in processed_metadata_dir.glob('GSE*.csv'):
        gse_id = file.stem  # Get GSE ID from filename
        valid_gse.add(gse_id)

df = df[df['accession'].isin(valid_gse)]
df

,title,description,organism,type,platform,accession,id,project,dataset,samples
208,Integrated analysis of cancer-related pathways...,DNA methylation was analyzed in human gastric ...,Homo sapiens,Methylation profiling by genome tiling array,GPL13534,GSE211704,200211704,NaN,NaN,50.0
210,Potential Methylation-regulated Genes and Path...,genome-wide profiling of DNA methylation (EPIC...,Homo sapiens,Methylation profiling by genome tiling array,GPL21145,GSE199747,200199747,NaN,NaN,8.0
211,Saliva methylome analysis of ART children that...,Follow-up study of 9 year old IVF children who...,Homo sapiens,Methylation profiling by array,GPL21145,GSE196432,200196432,NaN,NaN,120.0
213,Genome Wide DNA Methylation in the Neuronal Fr...,We analyzed the levels of DNA methylation in D...,Homo sapiens,Methylation profiling by genome tiling array,GPL21145,GSE156996,200156996,NaN,NaN,42.0
215,Epigenetic profiling of 16 look-alike human co...,The Illumina Infinium MethylationEPIC Beadchip...,Homo sapiens,Methylation profiling by array,GPL21145,GSE142302,200142302,NaN,NaN,32.0
...,...,...,...,...,...,...,...,...,...,...
51765,Large-scale analysis of the mouse transcriptome,High-throughput gene expression profiling has ...,Mus musculus,Expression profiling by array,GPL32,GSE97,200000097,NaN,GDS182,90.0
51766,wild type and aire -/- murine meduallary thymi...,"Mice used were B6/129 F2's, 3-5 weeks of age, ...",Mus musculus,Expression profiling by array,GPL81,GSE85,200000085,NaN,GDS167,6.0
51768,Multiplex three dimensional brain gene express...,Voxelation is a novel technology designed to p...,Mus musculus,Expression profiling by array,GPL69,GSE30,200000030,NaN,NaN,80.0
51769,NOD model of type 1 diabetes,We used high density oligonucleotide arrays to...,Mus musculus,Expression profiling by array,GPL24,GSE11,200000011,NaN,GDS10,28.0


In [21]:
df.to_csv(DATA / 'gse_all.csv', index=False)
df.to_parquet('data/gse_all.parquet')

# 3D coords to parquet

In [5]:
COORDS_DIR_PATH = DATA / '3dcoords'
COORDS_DIR_PATH_PARQUET = DATA / '3dcoords_parquet'
COORDS_DIR_PATH_PARQUET.mkdir(exist_ok=True)

In [6]:
# Convert all csv files in COORDS_DIR_PATH to parquet
import os
for filename in os.listdir(COORDS_DIR_PATH):
    if filename.endswith('.csv'):
        file_path = COORDS_DIR_PATH / filename
        df = pd.read_csv(file_path)
        parquet_file_path = COORDS_DIR_PATH_PARQUET / (filename[:-4] + '.parquet')
        df.to_parquet(parquet_file_path)

In [12]:
ANALYSIS_FILES = {}
for file in sorted(COORDS_DIR_PATH_PARQUET.glob("*.parquet")):
    name = file.stem  # e.g. 'human_dnam_acc_tsne_PhenoAge_coords'
    parts = name.split("_")
    
    # Example breakdown:
    # ['human', 'dnam', 'acc', 'tsne', 'PhenoAge', 'coords']
    method = None
    clock = None
    
    # Identify method and clock name
    if "tsne" in parts:
        method = "tsne"
    elif "umap_lite" in name:
        method = "umap_lite"
    elif "umap" in parts:
        method = "umap"

    # Extract clock name (the token before 'coords')
    for p in parts:
        if p.endswith("Age") or p in ["PedBE", "DunedinPACE", "DunedinPoAm", "ZhangAge"]:
            clock = p
            break

    if not (method and clock):
        print(f"⚠️ Skipping: {name}")
        continue

    key = f"{clock}_{method}"  # e.g. 'PhenoAge_tsne'
    ANALYSIS_FILES[key] = file.name

print("ANALYSIS_FILES = {")
for k, v in ANALYSIS_FILES.items():
    print(f"    '{k}': '{v}',")
print("}")

ANALYSIS_FILES = {
    'DunedinPACE_tsne': 'human_dnam_acc_tsne_DunedinPACE_coords.parquet',
    'DunedinPoAm_tsne': 'human_dnam_acc_tsne_DunedinPoAm_coords.parquet',
    'HannumAge_tsne': 'human_dnam_acc_tsne_HannumAge_coords.parquet',
    'HorvathAge_tsne': 'human_dnam_acc_tsne_HorvathAge_coords.parquet',
    'PedBE_tsne': 'human_dnam_acc_tsne_PedBE_coords.parquet',
    'PhenoAge_tsne': 'human_dnam_acc_tsne_PhenoAge_coords.parquet',
    'ZhangAge_tsne': 'human_dnam_acc_tsne_ZhangAge_coords.parquet',
    'DunedinPACE_umap': 'human_dnam_umap_DunedinPACE_coords.parquet',
    'DunedinPoAm_umap': 'human_dnam_umap_DunedinPoAm_coords.parquet',
    'HannumAge_umap': 'human_dnam_umap_HannumAge_coords.parquet',
    'HorvathAge_umap': 'human_dnam_umap_HorvathAge_coords.parquet',
    'PedBE_umap': 'human_dnam_umap_PedBE_coords.parquet',
    'PhenoAge_umap': 'human_dnam_umap_PhenoAge_coords.parquet',
    'ZhangAge_umap': 'human_dnam_umap_ZhangAge_coords.parquet',
    'DunedinPACE_umap_lite': 'hum